## Olivieri CRISPR Drug Screen — BAF Complex Extraction
### Author: Eleni Aretaki
### Date: 2026-03-31 (last edited)

### Purpose
Extract BAF complex genes from Olivieri CRISPR drug screen z-scores,
apply published significance thresholds, and visualize vulnerabilities
across compounds using MOA-annotated heatmaps.

Paper: https://doi.org/10.1016/j.cell.2020.05.040

Significance thresholds (Olivieri et al.):
    z-score > 6  → sensitization
    z-score < -3 → resistance

Plots generated:
    1. Full BAF heatmap
    2. Significant-only heatmap

In [ ]:
# -------------------------------
# Imports
# -------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

In [ ]:
# -------------------------------
# File paths & constants
# -------------------------------

# Select BAF genes
baf_genes = ['ARID1A', 'ARID1B', 'ARID2', 'SMARCA4', 'SMARCC1',
             'SMARCD1', 'PBRM1', 'PHF10', 'BRD7', 'BRD9']

# Set significance thresholds
UPPER_Z = 6
LOWER_Z = -3

# Read in z-scores dataset
oliv = pd.read_csv(
    '../olivieri_zscores.csv',
    sep=';'
)

# Read in drug annotation dataset
oliv_drugs = pd.read_csv(
    '../agents.csv',
    encoding='unicode_escape'
)

In [ ]:
# -------------------------------
# Preprocessing 
# -------------------------------
oliv_baf = oliv[
    oliv["Gene"].isin(baf_genes)
]

baf_df = oliv_baf.reset_index()

baf_df_long = baf_df.melt(
    id_vars='Gene',
    var_name='drug',
    value_name='z_score'
)

# Merge MOA
baf_df_long = baf_df_long.merge(
    oliv_drugs[['agent_id', 'mechanism_of_action']],
    left_on='drug',
    right_on='agent_id',
    how='left'
).drop(columns='agent_id')

custom_gene_order = [
    "ARID1A", "ARID1B", "ARID2",
    "BRD7", "PHF10", "PBRM1",
    "BRD9", "SMARCD1", "SMARCC1", "SMARCA4"
]

baf_df_long['Gene'] = pd.Categorical(
    baf_df_long['Gene'],
    categories=custom_gene_order,
    ordered=True
)

In [ ]:
# -------------------------------
# Plotting preparation
# -------------------------------

# ===============================
# CLEAN NAMES (important)
# ===============================
baf_df_long['drug'] = baf_df_long['drug'].str.strip()
baf_df_long['Gene'] = baf_df_long['Gene'].str.strip()
baf_df_long['moa'] = baf_df_long['mechanism_of_action'].fillna('Unknown')

# ===============================
# SORT BY MOA THEN DRUG
# ===============================
df_sorted = baf_df_long.sort_values(by=['mechanism_of_action', 'drug'])

# ===============================
# PIVOT FOR HEATMAP
# ===============================
heatmap_df = df_sorted.pivot(
    index='drug',
    columns='Gene',
    values='z_score'
)

# Enforce custom column order
heatmap_df = heatmap_df[custom_gene_order]

# Preserve sorted order explicitly
heatmap_df = heatmap_df.loc[df_sorted['drug'].unique()]

# ===============================
# CREATE MOA COLOR MAP
# ===============================

# Step 1: Choose fixed color for PARP
parp_color = extended_colors[11]

# Step 2: All MOAs
moas_heatmap = df_sorted['mechanism_of_action'].unique()

# Step 3: Build list of MOAs to assign dynamically (exclude PARP)
moas_to_assign = [moa for moa in moas_heatmap if moa != 'PARP inhibitor']

# Step 4: Remove parp_color from the color pool
other_colors = [c for c in extended_colors[12::1] if c != parp_color]

# Step 5: Build moa_colors dict
moa_colors = {'PARP inhibitor': parp_color}  # fixed mapping
for moa, color in zip(moas_to_assign, other_colors):
    moa_colors[moa] = color

# Step 6: MOA per drug
drug_to_moa = (
    df_sorted[['drug', 'mechanism_of_action']]
    .drop_duplicates()
    .set_index('drug')['mechanism_of_action']
    .to_dict()
)

In [ ]:
# -------------------------------
# Plotting 
# -------------------------------

# ===============================
# PLOT HEATMAP
# ===============================
plt.rcParams['font.family'] = 'Arial'

fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    heatmap_df,
    cmap='coolwarm',
    center=0,
    vmin=-3,
    vmax=3,
    linewidths=0.2,
    cbar_kws={'label': 'z-score', 'shrink': 0.5},
    ax=ax
)

# ===============================
# ADD SIGNIFICANCE STARS
# ===============================

for y, drug in enumerate(heatmap_df.index):
    for x, gene in enumerate(heatmap_df.columns):
        value = heatmap_df.iloc[y, x]

        if pd.notna(value) and (value > UPPER_Z or value < LOWER_Z):
            ax.text(
                x + 0.5,
                y + 0.5,
                '*',
                ha='center',
                va='center',
                color='black',
                fontsize=9,
                fontweight='bold'
            )

# ===============================
# ADD MOA COLOR BAR
# ===============================
for y, drug in enumerate(heatmap_df.index):
    ax.add_patch(
        mpatches.Rectangle(
            (0, y),
            -0.05,
            1,
            color=moa_colors[drug_to_moa.get(drug)],
            transform=ax.get_yaxis_transform(),
            clip_on=False
        )
    )

ax.tick_params(axis='y', pad=20)

# ===============================
# MOA LEGEND
# ===============================
handles = [
    mpatches.Patch(color=color, label=moa)
    for moa, color in moa_colors.items()
]

ax.legend(
    handles=handles,
    title='MOA',
    bbox_to_anchor=(1.3, 1),
    loc='upper left',
    frameon=False
)

# ===============================
# LABELS 
# ===============================
ax.set_xlabel('Knockout')
ax.set_ylabel('Drug')

plt.tight_layout()
plt.show()

fig.savefig("olivieri_full_heatmap.pdf", dpi=600)
plt.close()

In [ ]:
# -------------------------------
# Filter significant drugs only 
# -------------------------------
significant_mask = (
    (heatmap_df > UPPER_Z) |
    (heatmap_df < LOWER_Z)
)

significant_drugs = significant_mask.any(axis=1)

heatmap_sig = heatmap_df.loc[significant_drugs]

drug_to_moa_sig = {
    drug: drug_to_moa[drug]
    for drug in heatmap_sig.index
}

In [ ]:
# -------------------------------
# Significant drugs only heatmap
# -------------------------------
fig, ax = plt.subplots(figsize=(7, 3))

sns.heatmap(
    heatmap_sig,
    cmap='coolwarm',
    center=0,
    vmin=-3,
    vmax=3,
    linewidths=0.2,
    cbar_kws={'label': 'z-score', "shrink": 0.5},
    ax=ax
)

# MOA bar (subset)
for y, drug in enumerate(heatmap_sig.index):
    ax.add_patch(
        mpatches.Rectangle(
            (0, y),
            -0.05,
            1,
            color=moa_colors.get(drug_to_moa_sig.get(drug)),
            transform=ax.get_yaxis_transform(),
            clip_on=False
        )
    )

# Add stars for significant cells
for y, drug in enumerate(heatmap_sig.index):
    for x, gene in enumerate(heatmap_sig.columns):

        value = heatmap_sig.iloc[y, x]

        if pd.notna(value) and (
            value > UPPER_Z or value < LOWER_Z
        ):
            ax.text(
                x + 0.5,
                y + 0.5,
                '*',
                ha='center',
                va='center',
                color='black',
                fontsize=9,
                fontweight='bold'
            )

# MOA legend subset
present_moas = [
    drug_to_moa_sig[drug]
    for drug in heatmap_sig.index
]

present_moas = list(dict.fromkeys(present_moas))

handles = [
    mpatches.Patch(
        color=moa_colors[moa],
        label=moa
    )
    for moa in present_moas
]

ax.legend(
    handles=handles,
    title='MOA',
    bbox_to_anchor=(1.3, 1),
    loc='upper left',
    frameon=False
)

# Labels
ax.set_xlabel('Knockout')
ax.set_ylabel('Drug')

plt.tight_layout()
plt.show()

fig.savefig(
    "olivieri_significant_heatmap.pdf",
    dpi=600
)

plt.close()